# ABC2Vec — Notebook 08: Clustering Analysis

Evaluate whether ABC2Vec embeddings capture musically meaningful structure:

1. **Tune type clustering** — jig, reel, hornpipe, polka, slip jig, slide
2. **Mode clustering** — major, minor, dorian, mixolydian
3. **Key root clustering** — D, G, A, C, etc.
4. **UMAP / t-SNE visualization**

Metrics: Silhouette score, NMI, Adjusted Rand Index, linear probe accuracy.

In [ ]:
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, normalized_mutual_info_score, adjusted_rand_score,
    classification_report, confusion_matrix, accuracy_score, f1_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score

try:
    import umap
except ImportError:
    !pip install umap-learn
    import umap

from sklearn.manifold import TSNE

PROJECT_DIR = Path('/Volumes/LLModels/ABC2VEC')
BENCHMARK_DIR = PROJECT_DIR / 'data' / 'benchmark'
RESULTS_DIR = PROJECT_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR))

device = torch.device('cuda' if torch.cuda.is_available() else 
                       'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

## 8.1 Encode Test Set

In [ ]:
from abc2vec.model import ABC2VecModel, ABC2VecConfig
from abc2vec.tokenizer import ABCVocabulary, BarPatchifier

# Load vocab + model
vocab = ABCVocabulary()
vocab_path = PROJECT_DIR / 'data' / 'processed' / 'vocabulary.json'
if vocab_path.exists():
    vocab.load(vocab_path)

patchifier = BarPatchifier(vocab, max_bar_length=64, max_bars=64)

config = ABC2VecConfig(
    vocab_size=len(vocab),
    max_bar_len=64, max_bars=64,
    d_model=256, n_heads=8, n_layers=6,
    d_ff=1024, d_embed=128, dropout=0.1,
)
model = ABC2VecModel(config).to(device)

ckpt_path = PROJECT_DIR / 'checkpoints' / 'best_model.pt'
if ckpt_path.exists():
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded checkpoint")

model.eval()

# Load classification benchmark
class_df = pd.read_parquet(BENCHMARK_DIR / 'classification_test.parquet')
cluster_df = pd.read_parquet(BENCHMARK_DIR / 'clustering_test.parquet')
print(f"Classification set: {len(class_df)} tunes, {class_df['tune_type'].nunique()} types")
print(f"Clustering set: {len(cluster_df)} tunes, {cluster_df['mode'].nunique()} modes")

In [ ]:
@torch.no_grad()
def encode_tunes(abc_texts, model, patchifier, device, batch_size=64):
    """Encode ABC texts into embeddings."""
    model.eval()
    all_emb = []
    for i in tqdm(range(0, len(abc_texts), batch_size), desc="Encoding"):
        batch = abc_texts[i:i+batch_size]
        bar_ids_list, bar_mask_list = [], []
        for text in batch:
            bars = patchifier.patchify(text)
            ids, mask = patchifier.encode(bars)
            bar_ids_list.append(ids)
            bar_mask_list.append(mask)
        bar_ids = torch.stack(bar_ids_list).to(device)
        bar_mask = torch.stack(bar_mask_list).to(device)
        emb = model.get_embedding(bar_ids, bar_mask)
        all_emb.append(emb.cpu().numpy())
    return np.concatenate(all_emb, axis=0)

# Encode classification set
print("Encoding classification set...")
class_embeddings = encode_tunes(
    class_df['abc_body'].tolist(), model, patchifier, device
)
print(f"Shape: {class_embeddings.shape}")

# Encode clustering set
print("\nEncoding clustering set...")
cluster_embeddings = encode_tunes(
    cluster_df['abc_body'].tolist(), model, patchifier, device
)
print(f"Shape: {cluster_embeddings.shape}")

## 8.2 Tune Type Classification (Linear Probe)

In [ ]:
# Linear probe: train logistic regression on frozen embeddings
le_type = LabelEncoder()
y_type = le_type.fit_transform(class_df['tune_type'].values)

# Cross-validated accuracy
clf_type = LogisticRegression(max_iter=2000, C=1.0, multi_class='multinomial', solver='lbfgs')
cv_scores = cross_val_score(clf_type, class_embeddings, y_type, cv=5, scoring='accuracy')

print(f"Tune Type Linear Probe (5-fold CV):")
print(f"  Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  Per-fold: {[f'{s:.4f}' for s in cv_scores]}")

# Full fit for confusion matrix
clf_type.fit(class_embeddings, y_type)
y_pred = clf_type.predict(class_embeddings)

print(f"\nClassification Report (train):")
print(classification_report(y_type, y_pred, target_names=le_type.classes_))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_type, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_type.classes_, yticklabels=le_type.classes_, ax=ax,
            annot_kws={'size': 14})
ax.set_xlabel('Predicted', fontsize=14)
ax.set_ylabel('True', fontsize=14)
ax.set_title('Tune Type Classification — Confusion Matrix', fontsize=16, pad=20)
ax.tick_params(axis='both', labelsize=13)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'tune_type_confusion.pdf', bbox_inches='tight')
plt.savefig(RESULTS_DIR / 'tune_type_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 8.3 Mode Clustering (KMeans + Metrics)

In [ ]:
le_mode = LabelEncoder()
y_mode = le_mode.fit_transform(cluster_df['mode'].values)
n_modes = len(le_mode.classes_)

print(f"Modes: {list(le_mode.classes_)} ({n_modes} clusters)")

# KMeans clustering
kmeans_mode = KMeans(n_clusters=n_modes, random_state=42, n_init=10)
cluster_labels = kmeans_mode.fit_predict(cluster_embeddings)

# Metrics
sil = silhouette_score(cluster_embeddings, y_mode)
nmi = normalized_mutual_info_score(y_mode, cluster_labels)
ari = adjusted_rand_score(y_mode, cluster_labels)

print(f"\nMode Clustering Metrics:")
print(f"  Silhouette Score:          {sil:.4f}")
print(f"  NMI:                       {nmi:.4f}")
print(f"  Adjusted Rand Index:       {ari:.4f}")

# Linear probe for mode
clf_mode = LogisticRegression(max_iter=2000, C=1.0, multi_class='multinomial', solver='lbfgs')
cv_mode = cross_val_score(clf_mode, cluster_embeddings, y_mode, cv=5, scoring='accuracy')
print(f"  Linear probe accuracy:     {cv_mode.mean():.4f} ± {cv_mode.std():.4f}")

## 8.4 Key Root Clustering

In [ ]:
# Cluster by key root (D, G, A, etc.)
if 'key_root' in cluster_df.columns:
    # Filter to keys with enough samples
    key_counts = cluster_df['key_root'].value_counts()
    valid_keys = key_counts[key_counts >= 30].index.tolist()
    key_mask = cluster_df['key_root'].isin(valid_keys)
    
    key_df = cluster_df[key_mask].copy()
    key_emb = cluster_embeddings[key_mask.values]
    
    le_key = LabelEncoder()
    y_key = le_key.fit_transform(key_df['key_root'].values)
    n_keys = len(le_key.classes_)
    
    kmeans_key = KMeans(n_clusters=n_keys, random_state=42, n_init=10)
    key_clusters = kmeans_key.fit_predict(key_emb)
    
    sil_key = silhouette_score(key_emb, y_key)
    nmi_key = normalized_mutual_info_score(y_key, key_clusters)
    
    clf_key = LogisticRegression(max_iter=2000, C=1.0, multi_class='multinomial', solver='lbfgs')
    cv_key = cross_val_score(clf_key, key_emb, y_key, cv=5, scoring='accuracy')
    
    print(f"Key Root Clustering ({n_keys} keys):")
    print(f"  Keys: {list(le_key.classes_)}")
    print(f"  Silhouette: {sil_key:.4f}")
    print(f"  NMI:        {nmi_key:.4f}")
    print(f"  Probe acc:  {cv_key.mean():.4f} ± {cv_key.std():.4f}")
else:
    print("No key_root column available for clustering.")

## 8.5 UMAP Visualization by Tune Type

In [ ]:
# UMAP projection
print("Computing UMAP projection...")
reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.3,
                     metric='cosine', random_state=42)
umap_2d = reducer.fit_transform(class_embeddings)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# By tune type
tune_types = class_df['tune_type'].values
unique_types = sorted(set(tune_types))
type_colors = plt.cm.Set2(np.linspace(0, 1, len(unique_types)))

for i, tt in enumerate(unique_types):
    mask = tune_types == tt
    axes[0].scatter(umap_2d[mask, 0], umap_2d[mask, 1],
                    c=[type_colors[i]], label=tt, s=10, alpha=0.6)
axes[0].set_title('UMAP — Colored by Tune Type')
axes[0].legend(markerscale=3, loc='best')
axes[0].set_xticks([])
axes[0].set_yticks([])

# By mode (where available)
if 'mode' in class_df.columns:
    modes = class_df['mode'].fillna('unknown').values
    unique_modes = sorted(set(modes))
    mode_colors = plt.cm.tab10(np.linspace(0, 1, len(unique_modes)))
    
    for i, m in enumerate(unique_modes):
        mask = modes == m
        axes[1].scatter(umap_2d[mask, 0], umap_2d[mask, 1],
                        c=[mode_colors[i]], label=m, s=10, alpha=0.6)
    axes[1].set_title('UMAP — Colored by Mode')
    axes[1].legend(markerscale=3, loc='best')
    axes[1].set_xticks([])
    axes[1].set_yticks([])

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'umap_clustering.png', dpi=150, bbox_inches='tight')
plt.show()

## 8.6 t-SNE Visualization

In [ ]:
# t-SNE for comparison
print("Computing t-SNE projection...")
# Subsample if too large
max_tsne = 5000
if len(class_embeddings) > max_tsne:
    idx = np.random.RandomState(42).choice(len(class_embeddings), max_tsne, replace=False)
    tsne_emb = class_embeddings[idx]
    tsne_types = class_df['tune_type'].values[idx]
else:
    tsne_emb = class_embeddings
    tsne_types = class_df['tune_type'].values

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
tsne_2d = tsne.fit_transform(tsne_emb)

fig, ax = plt.subplots(figsize=(10, 8))
for i, tt in enumerate(unique_types):
    mask = tsne_types == tt
    ax.scatter(tsne_2d[mask, 0], tsne_2d[mask, 1],
               c=[type_colors[i]], label=tt, s=10, alpha=0.6)
ax.set_title('t-SNE — Colored by Tune Type')
ax.legend(markerscale=3, loc='best')
ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'tsne_clustering.png', dpi=150, bbox_inches='tight')
plt.show()

## 8.7 Summary Table

In [ ]:
results = {
    'Task': ['Tune Type Classification', 'Mode Clustering', 'Mode Linear Probe'],
    'Metric': ['5-fold CV Accuracy', 'Silhouette / NMI / ARI', '5-fold CV Accuracy'],
    'Result': [
        f"{cv_scores.mean():.4f} ± {cv_scores.std():.4f}",
        f"{sil:.4f} / {nmi:.4f} / {ari:.4f}",
        f"{cv_mode.mean():.4f} ± {cv_mode.std():.4f}",
    ]
}

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

results_df.to_csv(RESULTS_DIR / 'clustering_results.csv', index=False)
print(f"\nSaved to {RESULTS_DIR / 'clustering_results.csv'}")